# 09 — Executive Report Synthesis: Validation Stage Findings

Tổng kết điều hành các phát hiện kỹ thuật cốt lõi trong giai đoạn Validation; ma trận ý nghĩa thống kê Diebold-Mariano có hiệu chỉnh HLN và Holm; chuẩn bị cho Giai đoạn 3 (Test Evaluation).

### Project Guardrails & Execution Baseline:
- **Protocol Căn cứ:** `training_forecasting_evaluation_protocol_v1_0_21.md` và `spec_module_result_analysis_v2.md`
- **Zero Final Test Access Rule:** Chỉ nạp dữ liệu Validation (724 origins, 2018-05-01 đến 2020-04-23, H=7).
- **Purged Metrics Integrity:** Tuyệt đối không tính hoặc hiển thị CR-MAE@7, CR-Bias@7, RainDay-MAE, hay Bias.
- **Descriptive Cost Principle:** Số liệu tài nguyên chỉ dùng mô tả (PROP-07), không dùng để xếp hạng mô hình.


In [1]:
# Bootstrap root directory
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

ROOT_DIR = Path.cwd()
while not (ROOT_DIR / 'src').exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))


In [2]:
from src.analysis.loaders import load_all_validation_results, load_all_forecast_matrices
from src.analysis.ranking import compute_official_leaderboard, format_leaderboard_display
from src.analysis.significance import compute_champion_pairwise_dm
from src.analysis.aggregate import compute_family_aggregates

# 1. Family Aggregates
df_results = load_all_validation_results()
df_fam = compute_family_aggregates(df_results)
print("CANONICAL FAMILY PERFORMANCE (Macro vs Best):")
display(df_fam)

# 2. Champion Pairwise Diebold-Mariano Test
forecasts, y_true, _ = load_all_forecast_matrices()
df_ranked = compute_official_leaderboard(df_results)
top_5_scenarios = df_ranked['scenario_id'].head(5).tolist()

print(f"\nPAIRWISE DIEBOLD-MARIANO SIGNIFICANCE MATRIX (Top 5 Models: {top_5_scenarios}):")
df_dm = compute_champion_pairwise_dm(top_5_scenarios, forecasts, y_true, horizons=[1, 3, 7], p_adjust="holm")
display(df_dm)


CANONICAL FAMILY PERFORMANCE (Macro vs Best):


,family,n_scenarios,macro_mae,macro_rmse,macro_mase,best_scenario_id,best_model,best_mae,best_rmse,best_mase
0,Tabular ML Tweedie,8,3.183237,6.679412,0.859987,SC-30,lightgbm_tweedie_F2,3.148500,6.687300,0.850600
1,Two-Stage Hurdle,32,3.518360,7.000081,0.950512,SC-42,logistic_linear_stratA_F1,3.318662,6.690733,0.896562
2,Tabular ML Direct,16,3.471838,6.742275,0.937956,SC-13,lr_direct_F1,3.334200,6.654300,0.900800
3,Deep Neural,5,4.181227,11.035356,1.129590,SC-37,PatchTST,3.383765,7.190337,0.914150
4,Classical Statistical,4,3.707375,6.891450,1.001575,SC-04,AutoETS,3.661300,6.996400,0.989100
5,Intermittent Demand,5,3.685120,6.999980,0.995580,SC-08,CrostonSBA,3.663600,6.961200,0.989800
6,Heuristic Baseline,2,4.251600,9.174450,1.148600,SC-01,Naive,4.134900,8.995900,1.117100



PAIRWISE DIEBOLD-MARIANO SIGNIFICANCE MATRIX (Top 5 Models: ['SC-30', 'SC-34', 'SC-29', 'SC-35', 'SC-33']):


,horizon,model_1,model_2,DM_HLN_stat,p_value_raw,p_value_holm,significant_at_05
0,h=1,SC-30,SC-34,0.497247,0.619166,1.000000,False
1,h=1,SC-30,SC-29,-1.273494,0.203252,1.000000,False
2,h=1,SC-30,SC-35,-0.398886,0.690095,1.000000,False
3,h=1,SC-30,SC-33,-0.588053,0.556681,1.000000,False
4,h=1,SC-34,SC-29,-1.319859,0.187300,1.000000,False
5,h=1,SC-34,SC-35,-1.493219,0.135816,1.000000,False
6,h=1,SC-34,SC-33,-1.561075,0.118944,1.000000,False
7,h=1,SC-29,SC-35,0.507623,0.611873,1.000000,False
8,h=1,SC-29,SC-33,0.385108,0.700271,1.000000,False
9,h=1,SC-35,SC-33,-0.212549,0.831739,1.000000,False
